In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("../data/processed/chennai_house_price_prediction_cleaned.csv")

selected_features = joblib.load(
    "../models/selected_features.joblib"
)

X = df[selected_features]
y = df["SALES_PRICE"]

print("Number of selected features:", len(selected_features))

Number of selected features: 13


In [3]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Identify numerical and categorical features
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include="number"
).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['AREA', 'SALE_COND', 'PARK_FACIL', 'BUILDTYPE', 'UTILITY_AVAIL', 'STREET', 'MZZONE']
Numerical columns: ['INT_SQFT', 'N_BEDROOM', 'N_BATHROOM', 'N_ROOM', 'QS_OVERALL', 'COMMIS']


C:\Users\dhara\AppData\Local\Temp\ipykernel_5376\3031696571.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "numerical",
            "passthrough",
            numerical_cols
        )
    ]
)

In [6]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best CV R²:", grid_search.best_score_)

Best parameters: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV R²: 0.9813613363328807


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = best_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R² Score:", r2_score(y_test, y_pred))

MAE: 372770.35214486637
RMSE: 473921.16492108797
R² Score: 0.9829036782747774


In [8]:
import joblib

joblib.dump(
    best_model,
    "../models/tuned_random_forest.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [1]:
import joblib

model = joblib.load("../models/tuned_random_forest.joblib")

print("Model type:", type(model))

if hasattr(model, "feature_names_in_"):
    print("\nExpected input features:")
    print(model.feature_names_in_.tolist())

elif hasattr(model, "named_steps"):
    for name, step in model.named_steps.items():
        if hasattr(step, "feature_names_in_"):
            print(f"\nFeatures found in step: {name}")
            print(step.feature_names_in_.tolist())

Model type: <class 'sklearn.pipeline.Pipeline'>

Expected input features:
['AREA', 'INT_SQFT', 'N_BEDROOM', 'N_BATHROOM', 'N_ROOM', 'SALE_COND', 'PARK_FACIL', 'BUILDTYPE', 'UTILITY_AVAIL', 'STREET', 'MZZONE', 'QS_OVERALL', 'COMMIS']
